### 0. LABORATORY INITIALIZATION

### **Prerequisites for the Lab**

Before the session, ensure you have the environment set up.

1. Install [Ollama](https://ollama.com/) and download a lightweight model: `ollama run llama3` (or `llama3.1` or `mistral`).
2. Install the required Python libraries:

In [ ]:
# Step 1: Install system compression tools and dependencies
!sudo apt-get update && sudo apt-get install -y zstd

Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,764 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [99.9 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:13 http://security.ubuntu.com/ubuntu jammy

In [ ]:
# Step 2: Download and install the core Ollama environment binary
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
import os, subprocess, time

os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'

# Open a log file to catch the output so it doesn't fill the OS pipe
log_file = open('/content/ollama_server.log', 'w')

subprocess.Popen(
    ["ollama", "serve"],
    stdout=log_file,
    stderr=log_file
)

time.sleep(5)

In [ ]:
!ollama pull gemma4:latest

In [ ]:
!pip install langchain langchain-ollama langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.2 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
import os
import json
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.agents import initialize_agent, AgentType

In [ ]:
print("[SYSTEM] Hooking local reasoning engine via Ollama...")
# Temperature 0 enforces deterministic engineering metrics & format execution
llm = ChatOllama(
    model="gemma4:latest",
    base_url="http://localhost:11434",
    temperature=0
)

[SYSTEM] Hooking local reasoning engine via Ollama...


### 1. TOOL-USE PATTERN: APPLICANT TRACKING SYSTEM (ATS)

In [ ]:
@tool
def query_ats_database(candidate_name: str) -> str:
    """Searches the secure internal Applicant Tracking System (ATS) for a
    candidate's structured background and core technology stack profile."""
    # Simulating secure corporate database storage
    ats_vault = {
        "alice smith": "Profile: Alice Smith. Role: Data Scientist. Stack: Python, PyTorch, SQL. Experience: 4 years.",
        "bob jones": "Profile: Bob Jones. Role: Web Developer. Stack: JavaScript, React, CSS. Experience: 2 years."
    }
    key = candidate_name.lower().strip()
    return ats_vault.get(key, f"Candidate '{candidate_name}' not found in internal ATS records.")

### 2. REACT PATTERN: JOB DESCRIPTION EXTRACTION

In [ ]:
@tool
def fetch_job_mandates(job_id: str) -> str:
    """Retrieves explicit candidate qualification thresholds and engineering
    stack requirements from corporate job postings."""
    job_board = {
        "ds-100": "Job ID DS-100: Requires Python, PyTorch, and a minimum of 3+ years in Data Science.",
        "web-200": "Job ID WEB-200: Requires React, JavaScript, and 2+ years of frontend production."
    }
    return job_board.get(job_id.lower().strip(), f"Job Posting '{job_id}' not found.")

### 3. PLANNER PATTERN: STRATEGIC TASK DECOMPOSITION

In [ ]:
class RecruiterPlanner:
    def __init__(self, reasoning_engine):
        self.planner_prompt = ChatPromptTemplate.from_messages([
            ("system", """You are an Enterprise HR Workflow Architect.
            Your goal is to parse an ambiguous talent acquisition request and break it down into a strict, linear plan.
            Output your plan strictly as a JSON object using the following schema:
            {{
                "step_1": "Description of candidate verification task using query_ats_database tool",
                "step_2": "Description of skill benchmark verification using fetch_job_mandates tool"
            }}
            Do not run the steps. Only generate the plan structure."""),
            ("user", "Talent Goal: {hr_goal}")
        ])
        self.chain = self.planner_prompt | reasoning_engine

    def formulate_plan(self, hr_goal: str) -> dict:
        print("\n[PLANNER] Executing task decomposition loop...")
        raw_json_response = self.chain.invoke({"hr_goal": hr_goal}).content

        # Clean up possible LLM markdown decorations
        if "```json" in raw_json_response:
            raw_json_response = raw_json_response.split("```json")[1].split("```")[0]

        return json.loads(raw_json_response.strip())

### 4. EXECUTOR PATTERN: TACTICAL TOOL INTERFACING

In [ ]:
class RecruiterExecutor:
    def __init__(self, reasoning_engine):
        # Bundling custom Rest API tools into LangChain descriptor architecture
        self.tools = [query_ats_database, fetch_job_mandates]
        self.agent_runner = initialize_agent(
            tools=self.tools,
            llm=reasoning_engine,
            agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
            verbose=True,
            handle_parsing_errors=True
        )

    def execute_subtask(self, target_instruction: str) -> str:
        print(f"\n[EXECUTOR] Interfacing with environment for target instruction: '{target_instruction}'")
        return self.agent_runner.run(target_instruction)

### 5. EVALUATOR PATTERN: QUALITY ASSURANCE & REFLECTION

In [ ]:
class RecruitingComplianceEvaluator:
    def __init__(self, reasoning_engine):
        self.evaluator_prompt = ChatPromptTemplate.from_messages([
            ("system", """You are an Enterprise HR Compliance Officer and Executive Critic.
            Review the drafted outreach candidate summary text against corporate HR regulations.

            GRADING RUBRIC:
            1. Is the data factual to the candidate profile and requirements? (No hallucinations)
            2. Is the tone professional, objective, and compliant with talent brand parameters?

            If it fails the rubric, rewrite the text to be completely corporate-friendly.
            Format your final response with explicit 'STATUS', 'COMPLIANCE NOTES', and 'FINAL REPORT' sections."""),
            ("user", "Drafted Text to Evaluate:\n{draft_text}")
        ])
        self.chain = self.evaluator_prompt | reasoning_engine

    def evaluate_output(self, raw_report: str) -> str:
        print("\n[EVALUATOR] Running quality verification gate against grading rubric...")
        return self.chain.invoke({"draft_text": raw_report}).content

### 6. END-TO-END EXECUTION LOOP

In [ ]:
def run_recruiter_agent_pipeline(candidate: str, job_code: str):
    print(f"INITIATING AUTOMATED RECRUITMENT PIPELINE FOR: {candidate} -> {job_code}")

    # Instance declarations matching Slide 02
    planner = RecruiterPlanner(llm)
    executor = RecruiterExecutor(llm)
    evaluator = RecruitingComplianceEvaluator(llm)

    # Step A: Generate Abstract Map
    high_level_goal = f"Verify if candidate {candidate} possesses the mandatory capabilities for Job ID {job_code}."
    workflow_steps = planner.formulate_plan(high_level_goal)
    print(f"[PLANNER SUCCESS] Steps generated:\n{json.dumps(workflow_steps, indent=2)}")

    # Step B: Tactical Execution via Tool Interfacing
    step_1_output = executor.execute_subtask(workflow_steps["step_1"])
    step_2_output = executor.execute_subtask(workflow_steps["step_2"])

    # Step C: Synthesize for Verification Gate
    raw_synthesis = f"Candidate Status: {step_1_output}\nRole Benchmarks: {step_2_output}"

    # Step D: Reflection and Compliance Gate
    final_audit = evaluator.evaluate_output(raw_synthesis)

    print("RECRUITMENT PIPELINE AUDIT REPORT OUTPUT")
    print(final_audit)

In [ ]:
# --- RUN EXECUTION SCENARIO ---
run_recruiter_agent_pipeline(candidate="Alice Smith", job_code="DS-100")

INITIATING AUTOMATED RECRUITMENT PIPELINE FOR: Alice Smith -> DS-100

[PLANNER] Executing task decomposition loop...


/tmp/ipykernel_1461/2561056704.py:5: LangChainDeprecationWarning: Use `langchain.agents.create_agent` for new applications. It provides a more flexible agent factory with middleware support, structured output, and integration with LangGraph for persistence, streaming, and human-in-the-loop workflows. Migration guide: https://docs.langchain.com/oss/python/migrate/langchain-v1
  self.agent_runner = initialize_agent(


[PLANNER SUCCESS] Steps generated:
{
  "step_1": "Query the ATS database using candidate name 'Alice Smith' and Job ID 'DS-100' to retrieve her complete profile, work history, and self-reported skills for initial verification.",
  "step_2": "Fetch the mandatory skill set and experience requirements associated with Job ID DS-100 and cross-reference them against Alice Smith's verified profile data from Step 1 to determine capability gaps."
}

[EXECUTOR] Interfacing with environment for target instruction: 'Query the ATS database using candidate name 'Alice Smith' and Job ID 'DS-100' to retrieve her complete profile, work history, and self-reported skills for initial verification.'


> Entering new AgentExecutor chain...


/tmp/ipykernel_1461/2561056704.py:15: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  return self.agent_runner.run(target_instruction)


Action:
```json
{
  "action": "query_ats_database",
  "action_input": {
    "candidate_name": "Alice Smith"
  }
}
```
Observation: Profile: Alice Smith. Role: Data Scientist. Stack: Python, PyTorch, SQL. Experience: 4 years.
Thought:Action:
```json
{
  "action": "fetch_job_mandates",
  "action_input": {
    "job_id": "DS-100"
  }
}
```
Observation: Job ID DS-100: Requires Python, PyTorch, and a minimum of 3+ years in Data Science.
Thought:Thought: I have successfully queried both the candidate's profile and the job requirements using the provided tools. The ATS shows Alice Smith has 4 years of experience, proficiency in Python and PyTorch, and SQL skills. The job mandate (DS-100) requires Python, PyTorch, and a minimum of 3+ years in Data Science. Since her profile meets or exceeds all stated requirements, I can now provide the final verification summary to the user.
Action:
```json
{
  "action": "Final Answer",
  "action_input": "Initial Verification Summary for Alice Smith (Job ID DS